<a href="https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/03_pathway_enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Workshop 3 — Pathway enrichment analysis

**Companion to [Chapter 3: Pathway Enrichment Analysis & Visualization](https://omics.iotok.org/chapters/pathway-enrichment-protocol)**.

A gene list is hard to interpret. **Enrichment analysis** turns it into *pathways*. We use `gseapy`, which queries **Enrichr** (g:Profiler-style over-representation) and runs **GSEA**.

We load the DESeq2 results produced in Workshop 2 (already computed for you), so this notebook runs on its own.

In [ ]:
!pip install -q gseapy

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import gseapy as gp
RAW = 'https://raw.githubusercontent.com/muntakson/omics-atlas-workshop/main'
res = pd.read_csv(f'{RAW}/data/airway_deseq2_results.csv')
res = res.dropna(subset=['symbol','padj'])
print('genes with results:', len(res))
res.sort_values('padj').head()

## 1. Build the gene list
Over-representation analysis (ORA) asks: *are my upregulated genes enriched for any pathway more than chance?*

In [ ]:
sig = res[(res.padj < 0.05) & (res.log2FoldChange.abs() > 1)]
up_genes = sig[sig.log2FoldChange > 0]['symbol'].tolist()
print(len(up_genes), 'upregulated genes, e.g.:', up_genes[:8])

## 2. Over-representation with Enrichr
We test against **GO Biological Process** and **KEGG** gene-set libraries.

In [ ]:
enr = gp.enrichr(gene_list=up_genes,
                 gene_sets=['GO_Biological_Process_2021','KEGG_2021_Human'],
                 organism='human', outdir=None)
enr.results.sort_values('Adjusted P-value')[['Gene_set','Term','Adjusted P-value','Genes']].head(10)

## 3. Visualize — enriched-terms bar plot
The classic enrichment figure (Chapter 3): most significant pathways as a bar plot.

In [ ]:
from gseapy import dotplot
ax = dotplot(enr.results, column='Adjusted P-value', title='Enriched pathways (upregulated genes)',
             cmap='viridis_r', size=6, top_term=12, figsize=(5,7))
plt.show()

## 4. GSEA (no cutoff needed)
GSEA uses the **whole ranked list** (by fold-change) instead of a hard DEG cutoff — Chapter 3 → *Stage 2B*. It detects coordinated shifts even when no single gene is dramatic.

In [ ]:
rank = res[['symbol','log2FoldChange']].dropna().drop_duplicates('symbol')
rank = rank.sort_values('log2FoldChange', ascending=False).reset_index(drop=True)
pre = gp.prerank(rnk=rank, gene_sets='GO_Biological_Process_2021',
                 min_size=10, max_size=500, permutation_num=100, seed=7, outdir=None)
pre.res2d.sort_values('NES', ascending=False)[['Term','NES','FDR q-val']].head(8)

### ✏️ Your turn
Repeat the Enrichr over-representation test on the **down**regulated genes. Which pathways come up?

In [ ]:
# your code here


<details><summary>▶ Solution</summary>

In [ ]:
down_genes = sig[sig.log2FoldChange < 0]['symbol'].tolist()
gp.enrichr(gene_list=down_genes, gene_sets=['GO_Biological_Process_2021'],
           organism='human', outdir=None).results.sort_values('Adjusted P-value')[['Term','Adjusted P-value']].head(10)

</details>

## ✅ Recap
You turned a gene list into interpretable pathways with both ORA and GSEA.

**Next → [Workshop 4: Interpreting & visualizing enrichment](https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/04_interpreting_enrichment.ipynb)**.